# noisy: проверка на internal test

Проверяем уже обученную DINOv2 + geometry модель. Порог 0.35 зафиксирован на validation. Здесь ничего не обучаем и не подбираем.

## 1. Разбиение

In [1]:
%pip install -q catboost iterative-stratification

from pathlib import Path
import hashlib
import json
import zipfile

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'data/train.csv').is_file()), None)
if ROOT is None:
    ROOT = Path('/content/drive/MyDrive/Colab Notebooks/colab_upload/AI challenge main')
assert (ROOT / 'data/train.csv').is_file(), ROOT

package = ROOT / 'noisy_internal_test_assets.zip'
assert package.is_file(), f'Загрузи архив признаков в папку проекта: {package}'
with zipfile.ZipFile(package) as archive:
    for name in archive.namelist():
        if not (ROOT / name).is_file():
            archive.extract(name, ROOT)

labels = pd.read_csv(ROOT / 'data/train.csv')
targets = labels.columns.drop('item_id').tolist()
split = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=.15, random_state=42)
train_val_idx, test_idx = next(split.split(labels, labels[targets]))
test = labels.iloc[test_idx].reset_index(drop=True)
assert len(test) == 1344 and set(test.item_id).isdisjoint(labels.iloc[train_val_idx].item_id)
print('Internal test:', len(test), '| noisy:', int(test.noisy.sum()))

Mounted at /content/drive
Internal test: 1344 | noisy: 382


## 2. DINOv2 и геометрия

In [2]:
cache = ROOT / 'labels/simple/models/dinov2'
manifest = json.loads((cache / 'internal_test_embeddings.json').read_text(encoding='utf-8'))
assert manifest['model'] == 'dinov2_vits14'
assert manifest['item_ids'] == test.item_id.tolist()
assert manifest['labels_sha256'] == hashlib.sha256((ROOT / 'data/train.csv').read_bytes()).hexdigest()
views = np.load(cache / 'internal_test_embeddings.npy', mmap_mode='r')
assert views.shape == (1344, 6, 384) and np.isfinite(views).all()
visual = np.concatenate([views.mean(axis=1), views.max(axis=1)], axis=1)

config = json.loads((ROOT / 'archive/artifacts/catboost_features_config.json').read_text(encoding='utf-8'))
base_columns = config['feature_columns']
local_columns = ['pca_curvature_mean', 'pca_curvature_p90', 'pca_curvature_p95',
                 'normal_variation_p50', 'normal_variation_p90', 'normal_variation_p95',
                 'normal_variation_std', 'dihedral_over_10_share', 'dihedral_over_30_share']
base = pd.read_csv(ROOT / 'archive/artifacts/train_mesh_features_relative.csv').set_index('item_id')
local = pd.read_csv(ROOT / 'labels/lowpoly/artifacts/geometry_test_features.csv').set_index('item_id')
smooth = pd.read_csv(ROOT / 'labels/noisy/artifacts/smoothing_test_features.csv').set_index('item_id')
smooth_columns = [c for c in smooth if c.startswith('smooth_')]
assert len(base_columns) == 44 and len(local_columns) == 9 and len(smooth_columns) == 14
assert base.index.is_unique and local.index.is_unique and smooth.index.is_unique
assert set(local.index) == set(test.item_id) and set(smooth.index) == set(test.item_id)
geometry = base.join(local[local_columns]).join(smooth[smooth_columns])
geometry = geometry.loc[test.item_id, base_columns + local_columns + smooth_columns].to_numpy(dtype=np.float32)
assert geometry.shape == (1344, 67) and not np.isinf(geometry).any()
combined = np.concatenate([visual, geometry], axis=1)
assert combined.shape == (1344, 835)
print('DINOv2:', visual.shape, '| геометрия:', geometry.shape)

DINOv2: (1344, 768) | геометрия: (1344, 67)


## 3. Сохранённые модели и пороги

In [3]:
out = ROOT / 'labels/noisy/models/dinov2'
validation = pd.read_csv(out / 'metrics.csv').set_index('Experiment')
threshold_visual = float(validation.loc['D2_mean_max', 'Threshold'])
threshold_combined = float(validation.loc['D3_mean_max_geometry', 'Threshold'])
assert abs(threshold_visual - .34) < 1e-8
assert abs(threshold_combined - .35) < 1e-8

visual_model = CatBoostClassifier()
visual_model.load_model(str(out / 'D2_mean_max.cbm'))
combined_model = CatBoostClassifier()
combined_model.load_model(str(out / 'D3_mean_max_geometry.cbm'))
assert len(visual_model.feature_names_) == visual.shape[1]
assert len(combined_model.feature_names_) == combined.shape[1]
p_visual = visual_model.predict_proba(visual)[:, 1]
p_combined = combined_model.predict_proba(combined)[:, 1]
print('Пороги с validation:', threshold_visual, threshold_combined)

Пороги с validation: 0.34 0.35


## 4. Результат

In [4]:
y = test.noisy.to_numpy()

def score(probability, threshold):
    prediction = probability >= threshold
    tn, fp, fn, tp = confusion_matrix(y, prediction, labels=[0, 1]).ravel()
    return {'Precision': precision_score(y, prediction, zero_division=0),
            'Recall': recall_score(y, prediction, zero_division=0),
            'F1': f1_score(y, prediction, zero_division=0),
            'AP': average_precision_score(y, probability),
            'FP': int(fp), 'FN': int(fn), 'Threshold': threshold}

results = pd.DataFrame([
    {'Model': 'DINOv2 mean + max', **score(p_visual, threshold_visual)},
    {'Model': 'DINOv2 + geometry', **score(p_combined, threshold_combined)},
])
display(results.round(4))

predictions = test[['item_id', 'noisy', 'lowpoly', 'artifacts']].copy()
predictions['visual_probability'] = p_visual
predictions['combined_probability'] = p_combined
predictions['prediction'] = (p_combined >= threshold_combined).astype(int)
for label in ('lowpoly', 'artifacts'):
    group = predictions[(predictions.noisy == 1) & (predictions[label] == 1)]
    print(f'noisy + {label}: {len(group)} объектов, {int((group.prediction == 0).sum())} пропущено')

print('Сильные ложные срабатывания:')
display(predictions.query('noisy == 0 and prediction == 1').nlargest(5, 'combined_probability'))
print('Сильные пропуски:')
display(predictions.query('noisy == 1 and prediction == 0').nsmallest(5, 'combined_probability'))

results.to_csv(out / 'internal_test_metrics.csv', index=False)
predictions.to_csv(out / 'internal_test_probabilities.csv', index=False)
print('Сохранено:', out)

,Model,Precision,Recall,F1,AP,FP,FN,Threshold
0,DINOv2 mean + max,0.8229,0.8639,0.8429,0.9046,71,52,0.34
1,DINOv2 + geometry,0.8216,0.8560,0.8385,0.9151,71,55,0.35


noisy + lowpoly: 14 объектов, 10 пропущено
noisy + artifacts: 25 объектов, 7 пропущено
Сильные ложные срабатывания:


,item_id,noisy,lowpoly,artifacts,visual_probability,combined_probability,prediction
395,4e334ca1-54e4-4574-9b2e-26a8b88b9885,0,0,1,0.987993,0.984180,1
204,28c03b53-3279-4008-bc2d-46669db3b6af,0,0,0,0.953707,0.977951,1
999,c1c9249c-ee48-47b6-83d9-c9b6efbfe752,0,0,1,0.978979,0.962547,1
299,39f745ba-1121-4bc8-839d-7604baf2f7d1,0,0,1,0.958387,0.926042,1
562,6dfd8695-aa5c-40c0-9959-4db0344edb7c,0,1,1,0.894354,0.915409,1


Сильные пропуски:


,item_id,noisy,lowpoly,artifacts,visual_probability,combined_probability,prediction
1215,e3f03b51-eb07-41f5-92ae-1fc75fd4ea2e,1,1,0,0.006859,0.005397,0
772,95c5afb7-b1fc-4511-bce3-534ea4e0fd37,1,1,0,0.008804,0.006753,0
1064,cc90ab7c-78fc-44c3-a35d-1588ffb72938,1,1,0,0.007924,0.007819,0
785,982f64d1-b8ea-4a8b-bb3e-4b83aa2c730a,1,0,0,0.007393,0.012505,0
764,941af33d-c003-4988-a344-816cd1177f74,1,1,0,0.018551,0.020028,0


Сохранено: /content/drive/MyDrive/Colab Notebooks/colab_upload/AI challenge main/labels/noisy/models/dinov2


## Вывод

Сравниваем F1 с 0.8301 на validation у DINOv2 + geometry. Test использован только для этой проверки: модели и пороги не меняем по результату.